# NOVA Gold Lab — XAUUSD H1 CPU Research
Research-only CPU training. Authoritative source: Exness MT5 Demo on Windows through the tenant-isolated NOVA read-only Bridge and HTTPS Relay.\n\n**Dataset contract:** downloads genuine OHLC and MT5 `tick_volume`; validates UTC range, row count, ordering, duplicates, OHLC bounds, unexplained gaps, and SHA-256 before training. No volume is fabricated.\n\n**Secrets:** configure `NOVA_MT5_BRIDGE_URL`, `NOVA_MT5_BRIDGE_TOKEN`, and `NOVA_MT5_TENANT_ID` in Colab/Kaggle Secrets. Never store tokens in notebook cells, Git, or chat. No broker orders or live trading.

In [ ]:
from pathlib import Path
import os, json, hashlib, random, warnings
import numpy as np
import pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED)
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
OUTPUT_DIR = OUTPUT_ROOT / 'nova_training_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False  # Explicit human opt-in only after dataset and review gates pass.
RUN_LSTM = False  # LSTM also requires RUN_TRAINING=True.
HORIZON = 5
SEQUENCE_LENGTH = 32
LSTM_EPOCHS = 8
print('CPU-only mode:', True, '| output:', OUTPUT_DIR)

In [ ]:
def validate_ohlcv(frame):
    """Normalize and validate genuine OHLCV data; never invent missing volume."""
    data = frame.copy()
    data.columns = [str(column).strip().lower() for column in data.columns]
    aliases = {
        'timestamp_utc': 'timestamp', 'datetime': 'timestamp',
        'date': 'timestamp', 'time': 'timestamp',
        'tick_volume': 'volume',
    }
    rename = {old: new for old, new in aliases.items()
              if old in data.columns and new not in data.columns}
    data = data.rename(columns=rename)
    required = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(
            f"Missing required OHLCV columns {missing}. "
            "XAUUSD H1 source must provide genuine MT5 tick_volume; "
            "volume must not be fabricated."
        )
    raw_timestamp = data['timestamp']
    if pd.api.types.is_datetime64_any_dtype(raw_timestamp.dtype):
        data['timestamp'] = pd.to_datetime(
            raw_timestamp, utc=True, errors='coerce'
        )
    else:
        numeric_timestamp = pd.to_numeric(raw_timestamp, errors='coerce')
        if numeric_timestamp.notna().all():
            unit = 'ms' if float(numeric_timestamp.abs().median()) > 1e11 else 's'
            data['timestamp'] = pd.to_datetime(
                numeric_timestamp, unit=unit, utc=True, errors='coerce'
            )
        else:
            data['timestamp'] = pd.to_datetime(
                raw_timestamp, utc=True, errors='coerce'
            )
    if data['timestamp'].isna().any():
        raise ValueError('Invalid or missing timestamps found')
    for column in ['open', 'high', 'low', 'close', 'volume']:
        data[column] = pd.to_numeric(data[column], errors='coerce')
    if data[required].isna().any().any():
        raise ValueError('Missing or non-numeric OHLCV values found')
    if data['timestamp'].duplicated().any():
        raise ValueError('Duplicate timestamps found; resolve before training')
    if (data['volume'] < 0).any():
        raise ValueError('negative volume is not allowed')
    if (data['volume'] % 1 != 0).any():
        raise ValueError('tick_volume must be a non-negative integer')
    data['volume'] = data['volume'].astype('int64')
    invalid = (
        (data.high < data.low) |
        (data.open < data.low) | (data.open > data.high) |
        (data.close < data.low) | (data.close > data.high)
    )
    if invalid.any():
        raise ValueError('Invalid OHLC bounds')
    if not data['timestamp'].is_monotonic_increasing:
        raise ValueError('Timestamps must be strictly increasing in source order')
    return data.reset_index(drop=True)


In [ ]:
# Genuine Exness Demo MT5 history through the tenant-authenticated Windows Bridge.
# Configure these in Colab Secrets / Kaggle Secrets; never paste values into notebook cells.
import os, time, requests
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse

def read_secret(name):
    value = os.environ.get(name, "")
    if value: return value.strip()
    try:
        from google.colab import userdata
        return str(userdata.get(name) or "").strip()
    except Exception: pass
    try:
        from kaggle_secrets import UserSecretsClient
        return str(UserSecretsClient().get_secret(name) or "").strip()
    except Exception: return ""

BRIDGE_BASE_URL = read_secret("NOVA_MT5_BRIDGE_URL").rstrip("/")
BRIDGE_TOKEN = read_secret("NOVA_MT5_BRIDGE_TOKEN")
BRIDGE_TENANT_ID = read_secret("NOVA_MT5_TENANT_ID")
if not all([BRIDGE_BASE_URL, BRIDGE_TOKEN, BRIDGE_TENANT_ID]):
    raise RuntimeError("Set NOVA_MT5_BRIDGE_URL, NOVA_MT5_BRIDGE_TOKEN, NOVA_MT5_TENANT_ID in notebook Secrets.")
parsed_bridge = urlparse(BRIDGE_BASE_URL)
if parsed_bridge.scheme != "https" or parsed_bridge.hostname != "voom-mt5-bridge-relay.suwapat787.workers.dev" or parsed_bridge.path or parsed_bridge.query or parsed_bridge.username:
    raise ValueError("Use the approved HTTPS Cloudflare Relay origin.")
BRIDGE_HEADERS = {"Authorization": "Bearer " + BRIDGE_TOKEN, "X-VOOM-TENANT-ID": BRIDGE_TENANT_ID, "Accept": "application/json"}

SYMBOL = "XAUUSDm"  # Must exactly match the symbol name returned by the Exness Demo terminal.
TIMEFRAME = "H1"
START_UTC = datetime(2022, 7, 21, tzinfo=timezone.utc)
FINAL_OOS_START_UTC = datetime(2026, 7, 1, tzinfo=timezone.utc)
REQUESTED_END_UTC = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
END_UTC = min(REQUESTED_END_UTC, FINAL_OOS_START_UTC)  # Exclusive: final OOS is sealed and never downloaded for training.
CHUNK_DAYS = {"M1": 1, "M5": 5, "M15": 15, "M30": 30, "H1": 30, "H4": 90, "D1": 365}
if TIMEFRAME not in CHUNK_DAYS or START_UTC >= END_UTC:
    raise ValueError("Invalid timeframe or UTC date range.")

symbol_response = requests.get(BRIDGE_BASE_URL + "/api/symbols", params={"q": "XAUUSD", "limit": 100}, headers=BRIDGE_HEADERS, timeout=30)
symbol_response.raise_for_status()
available_symbols = [item["name"] for item in symbol_response.json().get("symbols", [])]
if SYMBOL not in available_symbols:
    raise ValueError(f"Exact symbol {SYMBOL!r} unavailable. Terminal symbols: {available_symbols}")

def fetch_history_chunk(start, end):
    params = {"symbol": SYMBOL, "timeframe": TIMEFRAME,
              "start_utc": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
              "end_utc": end.strftime("%Y-%m-%dT%H:%M:%SZ")}
    for attempt in range(3):
        try:
            response = requests.get(BRIDGE_BASE_URL + "/api/history", params=params, headers=BRIDGE_HEADERS, timeout=90)
            response.raise_for_status()
            payload = response.json()
            if payload.get("account_environment") != "EXNESS_DEMO":
                raise RuntimeError("Bridge did not confirm Exness Demo account environment.")
            if payload.get("mode") != "read-only" or payload.get("order_submission_enabled") is not False or payload.get("ordersEnabled") is not False or payload.get("liveTradingEnabled") is not False:
                raise RuntimeError("Bridge read-only safety contract failed.")
            if payload.get("symbol") != SYMBOL or payload.get("timeframe") != TIMEFRAME:
                raise RuntimeError("Bridge returned a different symbol/timeframe.")
            bars = payload.get("bars")
            if not isinstance(bars, list) or payload.get("count") != len(bars):
                raise RuntimeError("Bridge bar count does not match returned rows.")
            return bars
        except Exception:
            if attempt == 2: raise
            time.sleep(2 ** attempt)
    raise RuntimeError("History download failed.")

all_bars, chunk_count = [], 0
cursor = START_UTC
while cursor < END_UTC:
    chunk_end = min(cursor + timedelta(days=CHUNK_DAYS[TIMEFRAME]), END_UTC)
    bars = fetch_history_chunk(cursor, chunk_end)
    all_bars.extend(bars)
    chunk_count += 1
    print(f"Chunk {chunk_count}: {cursor.isoformat()} → {chunk_end.isoformat()} | {len(bars)} bars")
    cursor = chunk_end  # Bridge range is half-open [start, end); no chunk overlap.

if not all_bars:
    raise ValueError("Exness Demo Bridge returned no candles for this period.")
raw_df = pd.DataFrame(all_bars).rename(columns={"time": "timestamp_utc"})
df = validate_ohlcv(raw_df)
if len(df) != len(raw_df):
    raise ValueError("Normalization changed row count; refusing silent row loss.")
if df["timestamp"].duplicated().any() or not df["timestamp"].is_monotonic_increasing:
    raise ValueError("Timestamp order/uniqueness check failed.")
if (df["timestamp"] < pd.Timestamp(START_UTC)).any() or (df["timestamp"] >= pd.Timestamp(END_UTC)).any():
    raise ValueError("Returned timestamps fall outside the requested UTC interval.")
if int(df["volume"].sum()) <= 0:
    raise ValueError("MT5 tick_volume is empty or zero.")

# Weekend closures are expected. Large weekday gaps are recorded and block training until reviewed.
tf_minutes = {"M1": 1, "M5": 5, "M15": 15, "M30": 30, "H1": 60, "H4": 240, "D1": 1440}[TIMEFRAME]
interval = pd.Timedelta(minutes=tf_minutes)
def is_bounded_market_closure(
    previous, current, expected_interval, max_weekend_closure_hours=96,
    symbol="XAUUSDm",
):
    """Accept regular bars, documented XAU daily rollover, and bounded weekends."""
    gap = current - previous
    if gap == pd.Timedelta(0) or gap == expected_interval:
        return True

    # Exness uses UTC+0 servers; gold has a daily rollover pause. For this
    # XAUUSDm source, the observed seasonal H1 windows are 20-22 or 21-23 UTC.
    if (
        symbol == "XAUUSDm"
        and gap == pd.Timedelta(hours=2)
        and previous.date() == current.date()
        and previous.weekday() in (0, 1, 2, 3)
        and previous.minute == 0
        and current.minute == 0
        and (previous.hour, current.hour) in ((20, 22), (21, 23))
    ):
        return True

    if gap > pd.Timedelta(hours=max_weekend_closure_hours):
        return False
    cursor = previous.normalize() + pd.Timedelta(days=1)
    while cursor <= current.normalize():
        if cursor.weekday() >= 5:
            return True
        cursor += pd.Timedelta(days=1)
    return False


def detect_unexplained_gaps(
    timestamps, expected_interval, max_weekend_closure_hours=96, symbol="XAUUSDm"
):
    """Keep unrecognized weekday gaps and outages visible; never fill missing bars."""
    gaps = []
    for previous, current in zip(timestamps[:-1], timestamps[1:]):
        gap = current - previous
        if gap == expected_interval:
            continue
        if gap > expected_interval and is_bounded_market_closure(
            previous, current, expected_interval, max_weekend_closure_hours, symbol
        ):
            continue
        gaps.append({
            "after_utc": previous.isoformat(),
            "before_utc": current.isoformat(),
            "gap_minutes": int(gap.total_seconds() / 60),
        })
    return gaps


def detect_boundary_gaps(
    first_timestamp, last_timestamp, requested_start, requested_end,
    expected_interval, max_weekend_closure_hours=96, symbol="XAUUSDm",
):
    """Apply the same XAU rollover/weekend policy to both requested boundaries."""
    issues = []
    boundaries = (
        ("start", requested_start, first_timestamp),
        ("end", last_timestamp, requested_end),
    )
    for name, previous, current in boundaries:
        if not is_bounded_market_closure(
            previous, current, expected_interval, max_weekend_closure_hours, symbol
        ):
            issues.append({
                "boundary": name,
                "lag_hours": (current - previous).total_seconds() / 3600,
            })
    return issues
gap_rows = detect_unexplained_gaps(df["timestamp"], interval, symbol=SYMBOL)

# Apply the same 96-hour bounded-closure rule to both requested boundaries.
coverage_issues = detect_boundary_gaps(
    df["timestamp"].min(), df["timestamp"].max(),
    pd.Timestamp(START_UTC), pd.Timestamp(END_UTC), interval, symbol=SYMBOL
)
if len(df) < 1000:
    raise ValueError(f"Need at least 1,000 valid bars for this research baseline; received {len(df)}.")

# Serialize and hash in memory first. Never persist a Dataset that failed range/completeness checks.
csv_bytes = df.to_csv(index=False, float_format="%.17g").encode("utf-8")
sha256 = hashlib.sha256(csv_bytes).hexdigest()
validation_report = {
    "row_count": int(len(df)),
    "requested_start_utc": START_UTC.isoformat(),
    "requested_end_utc_exclusive": END_UTC.isoformat(),
    "actual_start_utc": df["timestamp"].min().isoformat(),
    "actual_end_utc": df["timestamp"].max().isoformat(),
    "unexplained_weekday_gaps": gap_rows,
    "coverage_issues": coverage_issues,
    "sha256_pre_save": sha256,
}
if gap_rows or coverage_issues:
    print(json.dumps(validation_report, indent=2))
    raise RuntimeError("Refusing to save Dataset: unexplained gaps or requested-period coverage issues were detected.")

DATA_DIR = Path("/kaggle/working/nova_exness_demo_dataset") if Path("/kaggle/working").exists() else Path("/content/nova_exness_demo_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)
dataset_path = DATA_DIR / f"{SYMBOL}_{TIMEFRAME}_exness_demo.csv"
data_path = dataset_path
dataset_path.write_bytes(csv_bytes)
sha256_on_disk = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
if sha256_on_disk != sha256:
    raise RuntimeError("On-disk SHA-256 differs from pre-save hash; refusing Dataset acceptance.")
manifest = {
    "dataset_name": dataset_path.name,
    "source": "Exness MT5 Demo via NOVA Windows Read-only Bridge",
    "bridge_origin": parsed_bridge.hostname,
    "account_environment": "EXNESS_DEMO",
    "symbol": SYMBOL,
    "timeframe": TIMEFRAME,
    "volume_semantics": "MT5 tick_volume; normalized to the volume column",
    "volume_unit": "tick_count",
    "requested_start_utc": START_UTC.isoformat(),
    "requested_end_utc_exclusive": END_UTC.isoformat(),
    "actual_start_utc": df["timestamp"].min().isoformat(),
    "actual_end_utc": df["timestamp"].max().isoformat(),
    "row_count": int(len(df)),
    "columns": list(df.columns),
    "chunk_count": chunk_count,
    "sha256": sha256_on_disk,
    "size_bytes": dataset_path.stat().st_size,
    "unexplained_weekday_gaps": gap_rows,
    "coverage_issues": coverage_issues,
    "completeness_status": "PASS",
    "training_eligible": True,
    "orders_submitted": False,
    "ordersEnabled": False,
    "live_trading_enabled": False,
    "promotion_approved": False,
}
(DATA_DIR / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("Verified Dataset:", dataset_path)


In [ ]:
# Data has already been downloaded and validated from the authenticated Exness Demo Bridge above.
data_path = dataset_path
if not data_path.is_file(): raise FileNotFoundError(data_path)
print('Using Exness Demo Bridge dataset:', data_path, '| rows:', len(df), '| UTC range:', df.timestamp.min(), '→', df.timestamp.max())

In [ ]:
def rsi(s, period=14):
    delta=s.diff(); gain=delta.clip(lower=0).rolling(period).mean(); loss=(-delta.clip(upper=0)).rolling(period).mean()
    rs=gain/loss.replace(0, np.nan); return 100-(100/(1+rs))
feat = pd.DataFrame(index=df.index)
feat['return_1']=df.close.pct_change()
feat['return_5']=df.close.pct_change(5)
feat['rsi_14']=rsi(df.close)
ema12=df.close.ewm(span=12, adjust=False).mean(); ema26=df.close.ewm(span=26, adjust=False).mean()
feat['macd']=ema12-ema26; feat['macd_signal']=feat.macd.ewm(span=9, adjust=False).mean()
feat['volatility_20']=feat.return_1.rolling(20).std()
feat['volume_ratio']=df.volume/(df.volume.rolling(20).mean().replace(0,np.nan))
feat['candle_range']=(df.high-df.low)/df.close.replace(0,np.nan)
future_return=df.close.shift(-HORIZON)/df.close-1
labels=(future_return>0).astype('int8').rename('label')
valid=feat.replace([np.inf,-np.inf],np.nan).notna().all(axis=1) & future_return.notna()
X=feat.loc[valid].reset_index(drop=True); y=labels.loc[valid].reset_index(drop=True)
timestamps=df.loc[valid,'timestamp'].reset_index(drop=True)
n=len(X); train_end=int(n*0.70); val_end=int(n*0.85)
# Purge HORIZON rows at split boundaries so training labels never look into the next partition.
train_limit=train_end-HORIZON; val_limit=val_end-HORIZON
if min(train_limit, val_limit-train_end, n-val_end) < 100: raise ValueError('Not enough rows after purging horizon overlap at split boundaries')
if y.iloc[:train_limit].nunique() < 2: raise ValueError('Training partition contains only one class; inspect dataset/label horizon')
print('Samples:', n, '| train:',train_limit,'| validation:',val_limit-train_end,'| test:',n-val_end,'| purged bars per boundary:',HORIZON)

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
def evaluate(name, y_true, prob):
    pred=(np.asarray(prob)>=0.5).astype(int)
    out={'model':name,'accuracy':float(accuracy_score(y_true,pred)),'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),
         'precision':float(precision_score(y_true,pred,zero_division=0)),'recall':float(recall_score(y_true,pred,zero_division=0)),
         'f1':float(f1_score(y_true,pred,zero_division=0))}
    out['roc_auc']=float(roc_auc_score(y_true,prob)) if len(np.unique(y_true))==2 else None
    return out
X_train,X_val,X_test=X.iloc[:train_limit],X.iloc[train_end:val_limit],X.iloc[val_end:]
y_train,y_val,y_test=y.iloc[:train_limit],y.iloc[train_end:val_limit],y.iloc[val_end:]
baseline=None; xgb=None; xgb_val=None; xgb_test=None
if RUN_TRAINING:
    dummy=DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
    baseline=evaluate('majority_baseline',y_test,dummy.predict_proba(X_test)[:,1])
    xgb=XGBClassifier(n_estimators=350,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,
                      objective='binary:logistic',eval_metric='logloss',n_jobs=2,tree_method='hist',random_state=SEED)
    xgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=False)
    xgb_val=evaluate('xgboost_validation',y_val,xgb.predict_proba(X_val)[:,1])
    xgb_test=evaluate('xgboost_test',y_test,xgb.predict_proba(X_test)[:,1])
    print(json.dumps({'baseline':baseline,'validation':xgb_val,'test':xgb_test},indent=2))
else:
    print('Training disabled: set RUN_TRAINING=True only after explicit human review.')


In [ ]:
lstm_test=None; lstm_val=None
if RUN_TRAINING and RUN_LSTM:
    import torch
    from torch import nn
    from sklearn.preprocessing import StandardScaler
    torch.manual_seed(SEED); torch.set_num_threads(2); device=torch.device('cpu')
    scaler=StandardScaler().fit(X_train)
    xs=scaler.transform(X).astype('float32'); ys=y.to_numpy(dtype='float32')
    def sequences(start,end):
        idx=np.arange(max(start,SEQUENCE_LENGTH-1),end)
        xx=np.stack([xs[i-SEQUENCE_LENGTH+1:i+1] for i in idx])
        return torch.tensor(xx),torch.tensor(ys[idx,None])
    tx,ty=sequences(SEQUENCE_LENGTH-1,train_limit); vx,vy=sequences(train_end,val_limit); qx,qy=sequences(val_end,n)
    class SmallLSTM(nn.Module):
        def __init__(self,n_features):
            super().__init__(); self.lstm=nn.LSTM(n_features,32,batch_first=True); self.head=nn.Sequential(nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
        def forward(self,z):
            h,_=self.lstm(z); return self.head(h[:,-1,:])
    net=SmallLSTM(xs.shape[1]).to(device); opt=torch.optim.Adam(net.parameters(),lr=0.001); loss_fn=nn.BCEWithLogitsLoss()
    best_state=None; best_loss=float('inf')
    for epoch in range(LSTM_EPOCHS):
        net.train(); order=torch.randperm(len(tx))
        for ids in order.split(256):
            opt.zero_grad(); loss=loss_fn(net(tx[ids]),ty[ids]); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): val_loss=float(loss_fn(net(vx),vy).item())
        if val_loss<best_loss: best_loss=val_loss; best_state={k:v.detach().clone() for k,v in net.state_dict().items()}
        print(f'epoch {epoch+1}/{LSTM_EPOCHS} validation_loss={val_loss:.5f}')
    net.load_state_dict(best_state); net.eval()
    with torch.no_grad():
        lstm_val=evaluate('lstm_validation',vy.numpy().ravel(),torch.sigmoid(net(vx)).numpy().ravel())
        lstm_test=evaluate('lstm_test',qy.numpy().ravel(),torch.sigmoid(net(qx)).numpy().ravel())
    torch.save({'state_dict':net.state_dict(),'feature_names':list(X.columns),'scaler_mean':scaler.mean_.tolist(),'scaler_scale':scaler.scale_.tolist(),'sequence_length':SEQUENCE_LENGTH},OUTPUT_DIR/'lstm_cpu.pt')
    print(json.dumps({'lstm_validation':lstm_val,'lstm_test':lstm_test},indent=2))

In [ ]:
import platform, sys
if RUN_TRAINING:
    xgb.save_model(OUTPUT_DIR/'xgboost_cpu.json')
if RUN_TRAINING:
    pd.DataFrame({'feature':X.columns,'importance':xgb.feature_importances_}).sort_values('importance',ascending=False).to_csv(OUTPUT_DIR/'xgboost_feature_importance.csv',index=False)
metrics={'training_executed':bool(RUN_TRAINING),'baseline_test':baseline,'xgboost_validation':xgb_val,'xgboost_test':xgb_test,'lstm_validation':lstm_val,'lstm_test':lstm_test,
         'samples':n,'train_rows':train_limit,'validation_rows':val_limit-train_end,'test_rows':n-val_end,'purged_horizon_rows_per_boundary':HORIZON,'horizon_bars':HORIZON,
         'test_start':str(timestamps.iloc[val_end]),'test_end':str(timestamps.iloc[-1]),'feature_names':list(X.columns),
         'cpu_only':True,'live_trading_enabled':False,'orders_submitted':False,'promotion_approved':False}
(OUTPUT_DIR/'metrics.json').write_text(json.dumps(metrics,indent=2))
hasher=hashlib.sha256()
with data_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024*1024), b''): hasher.update(chunk)
sha=hasher.hexdigest()
manifest={'dataset_file':data_path.name,'dataset_sha256':sha,'dataset_bytes':data_path.stat().st_size,'python':sys.version,
          'platform':platform.platform(),'random_seed':SEED,'run_mode':'kaggle-free-cpu','outputs':sorted(p.name for p in OUTPUT_DIR.iterdir()),
          'note':'Research candidate only. No model auto-promotion, broker writes, or live trading.'}
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved outputs:', sorted(p.name for p in OUTPUT_DIR.iterdir()))
if RUN_TRAINING:
    print('Compare XGBoost test accuracy against majority baseline:', xgb_test['accuracy'], 'vs', baseline['accuracy'])
else:
    print('No model fitting or model artifact creation occurred.')
print('Do not promote unless out-of-sample, walk-forward, costs, trade-count and risk gates all pass.')